# 38 — Activity Cliff Mining

Mine activity cliff pairs from ALL available bioactivity data:
PXR training + counter-assay + external sources from nb37.

Activity cliffs drive the test set — the test compounds are analogs of cliff hits,
so explicitly identifying cliff structure is critical for model improvement.

**Cliff definition:** Tanimoto(ECFP4) ≥ 0.5  AND  |ΔpEC50| ≥ 1.0  AND  SALI ≥ 2.0

where SALI = |ΔpEC50| / (1 - Tanimoto)

**Outputs:**
- `data/processed/cliff_labels.parquet` — per-training-compound cliff membership
- `data/processed/cliff_pairs.parquet` — all cliff pairs from PXR training
- `data/processed/cross_cliff_pairs.parquet` — PXR train vs external NR data

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pxr.data import load_train, load_test, load_counter
from pxr.chem import morgan_fp_batch, standardize_smiles, to_inchikey, bemis_murcko
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, FIGURES

TANIMOTO_THRESHOLD = 0.5
DELTA_PVAL_THRESHOLD = 1.0
SALI_THRESHOLD = 2.0

print('Setup complete.')

Setup complete.


## 1. Load All Data

Load PXR training, counter-assay, and external NR data from nb37 caches.
Standardize SMILES, compute InChIKeys, compute Morgan FP for all compounds.

In [2]:
# ── 1. Load all data ──────────────────────────────────────────────────────────
train   = load_train()
te      = load_test()
counter = load_counter()

print(f'PXR train:   {len(train):,} rows')
print(f'PXR test:    {len(te):,} rows')
print(f'Counter:     {len(counter):,} rows')

# Standardize training SMILES
print('\nStandardizing training SMILES...')
train['std_smiles'] = train['smiles'].map(standardize_smiles)
train['inchikey']   = train['std_smiles'].map(lambda s: to_inchikey(s) if s else None)
train = train.dropna(subset=['std_smiles', 'pec50']).reset_index(drop=True)
print(f'  Training after std: {len(train):,}')

te['std_smiles'] = te['smiles'].map(standardize_smiles)
te = te.dropna(subset=['std_smiles']).reset_index(drop=True)

# Load external data if available
external_dfs = []
ext_sources = [
    ('pubchem_pxr_aids.parquet',    'pec50',  'pubchem_pxr'),
    ('bindingdb_nr_data.parquet',   'pec50',  'bindingdb_nr'),
    ('chembl_nr_extended.parquet',  'pec50',  'chembl_nr'),
]
for fname, pval_col, src_label in ext_sources:
    cache_path = DATA_EXTERNAL / fname
    if cache_path.exists():
        try:
            df_ext = pd.read_parquet(cache_path)
            if pval_col in df_ext.columns and len(df_ext) > 0:
                df_ext = df_ext[df_ext[pval_col].notna()].copy()
                df_ext['source_label'] = src_label
                df_ext['pec50_for_cliff'] = df_ext[pval_col]
                # Ensure std_smiles exists
                if 'std_smiles' not in df_ext.columns:
                    smi_col = 'smiles'
                    df_ext['std_smiles'] = df_ext[smi_col].map(standardize_smiles)
                if 'inchikey' not in df_ext.columns:
                    df_ext['inchikey'] = df_ext['std_smiles'].map(
                        lambda s: to_inchikey(s) if s else None
                    )
                df_ext = df_ext.dropna(subset=['std_smiles', 'pec50_for_cliff'])
                external_dfs.append(df_ext)
                print(f'  Loaded {src_label}: {len(df_ext):,} rows')
        except Exception as e:
            print(f'  {fname}: load failed — {e}')
    else:
        print(f'  {fname}: not found — skipping (run nb37 first)')

print(f'\nExternal sources loaded: {len(external_dfs)}')

# Compute Morgan FP for training set
print('\nComputing Morgan FP for training set...')
train_fp = morgan_fp_batch(train['std_smiles'].tolist())  # (N_tr, 2048) uint8
print(f'Training FP matrix: {train_fp.shape}')

# Compute Morgan FP for test set
print('Computing Morgan FP for test set...')
te_fp = morgan_fp_batch(te['std_smiles'].tolist())  # (N_te, 2048) uint8
print(f'Test FP matrix: {te_fp.shape}')

PXR train:   4,139 rows
PXR test:    513 rows
Counter:     2,859 rows

Standardizing training SMILES...


  Training after std: 4,139


  Loaded bindingdb_nr: 5,690 rows
  Loaded chembl_nr: 11,496 rows

External sources loaded: 2

Computing Morgan FP for training set...


Training FP matrix: (4139, 2048)
Computing Morgan FP for test set...
Test FP matrix: (513, 2048)


## 2. Intra-dataset Cliff Mining (PXR Training)

For each pair of training compounds with Tanimoto ≥ 0.5 (ECFP4), compute SALI.
Vectorized: compute full pairwise Tanimoto using numpy bitwise operations.

For binary FPs: Tanimoto = |A ∩ B| / |A ∪ B|

In [3]:
# ── 2. Intra-dataset cliff mining (PXR training) ───────────────────────────────
def compute_pairwise_tanimoto(fp_matrix: np.ndarray) -> np.ndarray:
    """Compute (N, N) Tanimoto matrix from (N, D) binary uint8 FP matrix.
    Uses vectorized bitwise operations: Tan = intersection / union.
    Returns float32 to save memory.
    """
    fp = fp_matrix.astype(np.float32)  # bit sums via dot product
    # Bit counts per compound
    counts = fp.sum(axis=1)  # (N,)
    # Intersection: A & B bit count = fp_a · fp_b (for binary vectors)
    inter = fp @ fp.T   # (N, N)
    # Union = |A| + |B| - intersection
    union = counts[:, None] + counts[None, :] - inter
    # Avoid divide-by-zero
    tanimoto = np.where(union > 0, inter / union, 0.0).astype(np.float32)
    return tanimoto


N_tr = len(train)
print(f'Computing {N_tr}x{N_tr} pairwise Tanimoto matrix...')
# For N=4139 this is ~68M pairs, ~272 MB in float32 — manageable
tan_matrix = compute_pairwise_tanimoto(train_fp)
print(f'Tanimoto matrix computed: {tan_matrix.shape}  dtype={tan_matrix.dtype}')

y_train = train['pec50'].values

# Find all cliff pairs: upper triangle only (i < j)
print('\nMining cliff pairs...')
cliff_pairs = []

# Vectorized approach: get (i, j) indices where Tanimoto >= threshold
rows_i, cols_j = np.where(
    (tan_matrix >= TANIMOTO_THRESHOLD) & (np.triu(np.ones_like(tan_matrix, dtype=bool), k=1))
)
print(f'Pairs with Tanimoto >= {TANIMOTO_THRESHOLD}: {len(rows_i):,}')

for idx in range(len(rows_i)):
    i, j = int(rows_i[idx]), int(cols_j[idx])
    tan_ij     = float(tan_matrix[i, j])
    delta_pval = abs(float(y_train[i]) - float(y_train[j]))
    if delta_pval < DELTA_PVAL_THRESHOLD:
        continue
    sali = delta_pval / (1.0 - tan_ij + 1e-9)
    if sali < SALI_THRESHOLD:
        continue

    # Determine which is the cliff-active (higher pEC50)
    if y_train[i] >= y_train[j]:
        active_smi, active_pval = train['std_smiles'].iloc[i], y_train[i]
        inactive_smi, inactive_pval = train['std_smiles'].iloc[j], y_train[j]
        active_name = train['name'].iloc[i]
        inactive_name = train['name'].iloc[j]
    else:
        active_smi, active_pval = train['std_smiles'].iloc[j], y_train[j]
        inactive_smi, inactive_pval = train['std_smiles'].iloc[i], y_train[i]
        active_name = train['name'].iloc[j]
        inactive_name = train['name'].iloc[i]

    cliff_pairs.append({
        'smiles_a':          active_smi,
        'smiles_b':          inactive_smi,
        'name_a':            active_name,
        'name_b':            inactive_name,
        'pec50_a':           active_pval,
        'pec50_b':           inactive_pval,
        'tanimoto':          tan_ij,
        'delta_pec50':       delta_pval,
        'sali':              sali,
        'cliff_active_smiles':   active_smi,
        'cliff_inactive_smiles': inactive_smi,
    })

cliff_pairs_df = pd.DataFrame(cliff_pairs)
if len(cliff_pairs_df) > 0:
    cliff_pairs_df.to_parquet(DATA_PROCESSED / 'cliff_pairs.parquet', index=False)

print(f'\nCliff pairs found: {len(cliff_pairs_df):,}')
if len(cliff_pairs_df) > 0:
    print(f'SALI: mean={cliff_pairs_df["sali"].mean():.2f}  max={cliff_pairs_df["sali"].max():.2f}')
    print(f'Tanimoto: mean={cliff_pairs_df["tanimoto"].mean():.3f}  max={cliff_pairs_df["tanimoto"].max():.3f}')
    print(f'DeltapEC50: mean={cliff_pairs_df["delta_pec50"].mean():.2f}  max={cliff_pairs_df["delta_pec50"].max():.2f}')
    print(f'\nTop 10 cliff pairs by SALI:')
    print(cliff_pairs_df.nlargest(10, 'sali')[['name_a', 'name_b', 'pec50_a', 'pec50_b', 'tanimoto', 'sali']].to_string(index=False))
print(f'\nSaved to {DATA_PROCESSED}/cliff_pairs.parquet')

Computing 4139x4139 pairwise Tanimoto matrix...


Tanimoto matrix computed: (4139, 4139)  dtype=float32

Mining cliff pairs...
Pairs with Tanimoto >= 0.5: 549

Cliff pairs found: 149
SALI: mean=8187923.34  max=1220000000.00
Tanimoto: mean=0.568  max=1.000
DeltapEC50: mean=1.65  max=3.32

Top 10 cliff pairs by SALI:
        name_a         name_b  pec50_a  pec50_b  tanimoto         sali
OADMET-0003733 OADMET-0003684     3.20     1.98  1.000000 1.220000e+09
OADMET-0001916 OADMET-0001945     4.95     1.81  0.722222 1.130400e+01
OADMET-0005354 OADMET-0003285     4.45     1.73  0.689655 8.764445e+00
OADMET-0001977 OADMET-0002297     5.42     2.20  0.612245 8.304211e+00
OADMET-0001934 OADMET-0002279     5.29     1.97  0.574468 7.802000e+00
OADMET-0002884 OADMET-0002217     5.21     2.05  0.578947 7.505000e+00
OADMET-0002753 OADMET-0003355     6.15     2.96  0.553846 7.150000e+00
OADMET-0001951 OADMET-0002253     5.95     3.05  0.586957 7.021052e+00
OADMET-0001943 OADMET-0002265     5.52     2.94  0.630435 6.981177e+00
OADMET-0001894 OADMET-0

## 3. Compound-level Cliff Labels

For each training compound, aggregate over all cliff pairs it belongs to:
- `is_cliff_member`: appears in any cliff pair
- `n_cliff_pairs`: number of cliff pairs
- `cliff_role`: +1=cliff-active, -1=cliff-inactive, 0=not member
- `max_cliff_delta`: max |ΔpEC50| across its cliff pairs

Saved to `data/processed/cliff_labels.parquet` indexed by compound `name`.

In [4]:
# ── 3. Compound-level cliff labels ────────────────────────────────────────────
# Initialize all compounds as non-cliff
cliff_labels = train[['name', 'smiles', 'std_smiles', 'pec50']].copy()
cliff_labels['is_cliff_member'] = False
cliff_labels['n_cliff_pairs']   = 0
cliff_labels['cliff_role']      = 0
cliff_labels['max_cliff_delta'] = 0.0

if len(cliff_pairs_df) > 0:
    # Count active membership
    active_counts = cliff_pairs_df.groupby('name_a').agg(
        n_active_pairs=('delta_pec50', 'count'),
        max_active_delta=('delta_pec50', 'max'),
    )
    inactive_counts = cliff_pairs_df.groupby('name_b').agg(
        n_inactive_pairs=('delta_pec50', 'count'),
        max_inactive_delta=('delta_pec50', 'max'),
    )

    cliff_labels = cliff_labels.set_index('name')

    # Active members (cliff-active role)
    for name, row in active_counts.iterrows():
        if name in cliff_labels.index:
            cliff_labels.loc[name, 'is_cliff_member'] = True
            cliff_labels.loc[name, 'n_cliff_pairs']   += row['n_active_pairs']
            cliff_labels.loc[name, 'cliff_role']       = 1   # cliff-active
            cliff_labels.loc[name, 'max_cliff_delta']  = max(
                cliff_labels.loc[name, 'max_cliff_delta'],
                row['max_active_delta']
            )

    # Inactive members (cliff-inactive role — may override to mixed later)
    for name, row in inactive_counts.iterrows():
        if name in cliff_labels.index:
            cliff_labels.loc[name, 'is_cliff_member'] = True
            cliff_labels.loc[name, 'n_cliff_pairs']   += row['n_inactive_pairs']
            # If already active, keep +1 (compound can be active vs one and inactive vs another)
            if cliff_labels.loc[name, 'cliff_role'] == 0:
                cliff_labels.loc[name, 'cliff_role'] = -1  # cliff-inactive
            cliff_labels.loc[name, 'max_cliff_delta'] = max(
                cliff_labels.loc[name, 'max_cliff_delta'],
                row['max_inactive_delta']
            )

    cliff_labels = cliff_labels.reset_index()

cliff_labels.to_parquet(DATA_PROCESSED / 'cliff_labels.parquet', index=False)

n_cliff = cliff_labels['is_cliff_member'].sum()
n_active_role = (cliff_labels['cliff_role'] == 1).sum()
n_inactive_role = (cliff_labels['cliff_role'] == -1).sum()
print(f'Cliff member compounds: {n_cliff:,} / {len(cliff_labels):,} ({n_cliff/len(cliff_labels)*100:.1f}%)')
print(f'  Cliff-active role (+1): {n_active_role:,}')
print(f'  Cliff-inactive role (-1): {n_inactive_role:,}')
print(f'\nTop 10 cliff members by n_cliff_pairs:')
print(
    cliff_labels[cliff_labels['is_cliff_member']]
    .nlargest(10, 'n_cliff_pairs')[['name', 'pec50', 'cliff_role', 'n_cliff_pairs', 'max_cliff_delta']]
    .to_string(index=False)
)
print(f'\nSaved to {DATA_PROCESSED}/cliff_labels.parquet')

Cliff member compounds: 248 / 4,139 (6.0%)
  Cliff-active role (+1): 133
  Cliff-inactive role (-1): 115

Top 10 cliff members by n_cliff_pairs:
          name  pec50  cliff_role  n_cliff_pairs  max_cliff_delta
OADMET-0002265   2.94          -1              4            2.580
OADMET-0002279   1.97          -1              3            3.320
OADMET-0002006   5.19           1              3            3.270
OADMET-0001993   2.98          -1              3            1.765
OADMET-0005186   4.40          -1              2            1.930
OADMET-0004815   4.80           1              2            1.780
OADMET-0003735   3.25           1              2            1.490
OADMET-0003731   4.74           1              2            2.910
OADMET-0003721   1.99          -1              2            1.300
OADMET-0003664   3.40          -1              2            1.550

Saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed/cliff_labels.parquet


## 4. Cross-dataset Cliff Mining (Training vs External NR Data)

Find pairs where one compound is from PXR training and another from external NR data,
with Tanimoto ≥ 0.5 and |ΔpEC50| ≥ 1.0.

These cross-dataset structural analogs with divergent activity represent meaningful
biological context — different assay conditions, different receptor subtypes.

In [5]:
# ── 4. Cross-dataset cliff mining (training vs external NR data) ───────────────
CROSS_CLIFF_CACHE = DATA_PROCESSED / 'cross_cliff_pairs.parquet'

if not external_dfs:
    print('No external data available — skipping cross-dataset cliff mining')
    print('Run nb37 first to fetch external data.')
    cross_cliff_df = pd.DataFrame()
else:
    # Combine all external sources
    ext_combined = pd.concat(external_dfs, ignore_index=True)

    # Remove compounds already in training (by InChIKey)
    train_ik_set = set(train['inchikey'].dropna())
    ext_combined = ext_combined[
        ~ext_combined['inchikey'].isin(train_ik_set)
    ].drop_duplicates(subset=['inchikey']).reset_index(drop=True)
    print(f'External compounds (not in train): {len(ext_combined):,}')

    if len(ext_combined) == 0:
        print('No novel external compounds after deduplication')
        cross_cliff_df = pd.DataFrame()
    else:
        # Compute FPs for external compounds
        print('Computing Morgan FP for external compounds...')
        ext_fp = morgan_fp_batch(ext_combined['std_smiles'].tolist())
        ext_y  = ext_combined['pec50_for_cliff'].values
        print(f'External FP matrix: {ext_fp.shape}')

        # Compute cross-dataset Tanimoto: (N_tr, N_ext)
        # Process in batches to avoid memory issues
        BATCH_SIZE = 500
        print(f'\nComputing cross Tanimoto in batches of {BATCH_SIZE}...')

        cross_cliff_rows = []
        tr_fp_float = train_fp.astype(np.float32)
        ext_fp_float = ext_fp.astype(np.float32)
        tr_counts = tr_fp_float.sum(axis=1)
        ext_counts = ext_fp_float.sum(axis=1)

        for ext_start in range(0, len(ext_combined), BATCH_SIZE):
            ext_end = min(ext_start + BATCH_SIZE, len(ext_combined))
            ext_batch = ext_fp_float[ext_start:ext_end]
            ext_cnt_batch = ext_counts[ext_start:ext_end]

            # Cross intersection: (N_tr, batch)
            inter_cross = tr_fp_float @ ext_batch.T
            union_cross = tr_counts[:, None] + ext_cnt_batch[None, :] - inter_cross
            tan_cross = np.where(union_cross > 0, inter_cross / union_cross, 0.0)

            # Find pairs above threshold
            i_ids, j_ids = np.where(tan_cross >= TANIMOTO_THRESHOLD)
            for k in range(len(i_ids)):
                i = int(i_ids[k])
                j_local = int(j_ids[k])
                j_global = ext_start + j_local

                tan_ij    = float(tan_cross[i, j_local])
                tr_pval   = float(y_train[i])
                ext_pval  = float(ext_y[j_global])
                delta     = abs(tr_pval - ext_pval)

                if delta >= DELTA_PVAL_THRESHOLD:
                    sali = delta / (1.0 - tan_ij + 1e-9)
                    cross_cliff_rows.append({
                        'train_name':     train['name'].iloc[i],
                        'train_smiles':   train['std_smiles'].iloc[i],
                        'train_pec50':    tr_pval,
                        'ext_smiles':     ext_combined['std_smiles'].iloc[j_global],
                        'ext_pec50':      ext_pval,
                        'ext_source':     ext_combined['source_label'].iloc[j_global] if 'source_label' in ext_combined.columns else 'external',
                        'tanimoto':       tan_ij,
                        'delta_pec50':    delta,
                        'sali':           sali,
                    })

            if (ext_start // BATCH_SIZE) % 5 == 0:
                print(f'  Processed {ext_end:,}/{len(ext_combined):,} external compounds...')

        cross_cliff_df = pd.DataFrame(cross_cliff_rows)
        if len(cross_cliff_df) > 0:
            cross_cliff_df.to_parquet(CROSS_CLIFF_CACHE, index=False)
            print(f'\nCross-dataset cliff pairs: {len(cross_cliff_df):,}')
            print(cross_cliff_df.groupby('ext_source')['delta_pec50'].describe().round(2).to_string())
        else:
            print('No cross-dataset cliff pairs found')
            cross_cliff_df = pd.DataFrame()
            cross_cliff_df.to_parquet(CROSS_CLIFF_CACHE, index=False)

print(f'\nSaved to {CROSS_CLIFF_CACHE}')

External compounds (not in train): 11,185
Computing Morgan FP for external compounds...


External FP matrix: (11185, 2048)

Computing cross Tanimoto in batches of 500...
  Processed 500/11,185 external compounds...


  Processed 3,000/11,185 external compounds...


  Processed 5,500/11,185 external compounds...


  Processed 8,000/11,185 external compounds...


  Processed 10,500/11,185 external compounds...

Cross-dataset cliff pairs: 524
              count  mean  std   min   25%   50%   75%   max
ext_source                                                  
bindingdb_nr  193.0  3.27  1.6  1.02  2.08  3.02  3.76  8.81
chembl_nr     331.0  2.46  1.2  1.00  1.60  2.25  2.92  8.97

Saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\cross_cliff_pairs.parquet


## 5. Test Set Cliff Proximity

For each test compound, find its nearest training neighbor and report if that neighbor
is a cliff compound. Rank test compounds by cliff_proximity_score = max Tanimoto
to any cliff member in the training set.

In [6]:
# ── 5. Test set cliff proximity ───────────────────────────────────────────────
cliff_member_mask = cliff_labels.set_index('name')['is_cliff_member']
cliff_indices = [i for i, name in enumerate(train['name']) if cliff_member_mask.get(name, False)]
print(f'Cliff member compounds in training: {len(cliff_indices):,}')

if cliff_indices:
    cliff_fp = train_fp[cliff_indices]   # (n_cliff, 2048)
    cliff_pec50 = y_train[cliff_indices]

    # Cross Tanimoto: test vs all training, and test vs cliff members
    te_fp_float  = te_fp.astype(np.float32)
    tr_fp_float  = train_fp.astype(np.float32)
    cl_fp_float  = cliff_fp.astype(np.float32)

    te_counts  = te_fp_float.sum(axis=1)
    tr_counts  = tr_fp_float.sum(axis=1)
    cl_counts  = cl_fp_float.sum(axis=1)

    # Test vs all training
    inter_te_tr = te_fp_float @ tr_fp_float.T
    union_te_tr = te_counts[:, None] + tr_counts[None, :] - inter_te_tr
    tan_te_tr   = np.where(union_te_tr > 0, inter_te_tr / union_te_tr, 0.0)

    # Test vs cliff members only
    inter_te_cl = te_fp_float @ cl_fp_float.T
    union_te_cl = te_counts[:, None] + cl_counts[None, :] - inter_te_cl
    tan_te_cl   = np.where(union_te_cl > 0, inter_te_cl / union_te_cl, 0.0)

    proximity_rows = []
    for ti in range(len(te)):
        # Nearest training neighbor
        nn_idx   = int(np.argmax(tan_te_tr[ti]))
        nn_tan   = float(tan_te_tr[ti, nn_idx])
        nn_name  = train['name'].iloc[nn_idx]
        nn_pec50 = float(y_train[nn_idx])
        nn_is_cliff = cliff_member_mask.get(nn_name, False)

        # Max Tanimoto to any cliff member
        cliff_prox = float(tan_te_cl[ti].max())

        # Nearest cliff member
        nearest_cliff_idx = int(np.argmax(tan_te_cl[ti]))
        nearest_cliff_name = train['name'].iloc[cliff_indices[nearest_cliff_idx]]
        nearest_cliff_pec50 = float(cliff_pec50[nearest_cliff_idx])

        proximity_rows.append({
            'test_name':            te['name'].iloc[ti],
            'nn_train_name':        nn_name,
            'nn_tanimoto':          nn_tan,
            'nn_pec50':             nn_pec50,
            'nn_is_cliff':          bool(nn_is_cliff),
            'cliff_proximity_score': cliff_prox,
            'nearest_cliff_name':   nearest_cliff_name,
            'nearest_cliff_pec50':  nearest_cliff_pec50,
        })

    prox_df = pd.DataFrame(proximity_rows)
    prox_df.to_parquet(DATA_PROCESSED / 'test_cliff_proximity.parquet', index=False)

    print(f'\nTest set cliff proximity:')
    print(f'  Test compounds with nearest neighbor being cliff member: '
          f'{prox_df["nn_is_cliff"].sum():,} / {len(prox_df):,}')
    print(f'  Cliff proximity score (max Tan to cliff member):')
    print(f'    Mean:   {prox_df["cliff_proximity_score"].mean():.3f}')
    print(f'    Median: {prox_df["cliff_proximity_score"].median():.3f}')
    print(f'    > 0.7:  {(prox_df["cliff_proximity_score"] > 0.7).sum():,}')
    print(f'    > 0.5:  {(prox_df["cliff_proximity_score"] > 0.5).sum():,}')

    print(f'\nTop 10 test compounds most similar to cliff members:')
    print(
        prox_df.nlargest(10, 'cliff_proximity_score')[
            ['test_name', 'cliff_proximity_score', 'nearest_cliff_name', 'nearest_cliff_pec50']
        ].to_string(index=False)
    )
else:
    print('No cliff members found — skipping test set proximity analysis')
    prox_df = pd.DataFrame()

print(f'\nSaved to {DATA_PROCESSED}/test_cliff_proximity.parquet')

Cliff member compounds in training: 248



Test set cliff proximity:
  Test compounds with nearest neighbor being cliff member: 37 / 513
  Cliff proximity score (max Tan to cliff member):
    Mean:   0.296
    Median: 0.274
    > 0.7:  1
    > 0.5:  32

Top 10 test compounds most similar to cliff members:
     test_name  cliff_proximity_score nearest_cliff_name  nearest_cliff_pec50
OADMET-0006576               0.740000     OADMET-0003341                 6.09
OADMET-0006508               0.698113     OADMET-0003341                 6.09
OADMET-0006121               0.671875     OADMET-0002797                 5.84
OADMET-0006277               0.655738     OADMET-0002797                 5.84
OADMET-0006614               0.614035     OADMET-0003341                 6.09
OADMET-0006543               0.614035     OADMET-0005092                 4.73
OADMET-0006359               0.614035     OADMET-0003341                 6.09
OADMET-0006525               0.607143     OADMET-0006008                 4.81
OADMET-0006145               0.60

## 6. Summary Plots

- SALI score distribution across all cliff pairs
- Test set proximity to cliff members

In [7]:
# ── 6. Summary plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) SALI score distribution
ax = axes[0]
if len(cliff_pairs_df) > 0:
    ax.hist(cliff_pairs_df['sali'].clip(0, 30), bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(SALI_THRESHOLD, color='red', linestyle='--', label=f'threshold={SALI_THRESHOLD}')
    ax.set_xlabel('SALI Score')
    ax.set_ylabel('Count')
    ax.set_title(f'SALI Distribution\n({len(cliff_pairs_df):,} cliff pairs)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No cliff pairs', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('SALI Distribution')

# (b) Delta pEC50 vs Tanimoto scatter
ax = axes[1]
if len(cliff_pairs_df) > 0:
    scatter = ax.scatter(
        cliff_pairs_df['tanimoto'],
        cliff_pairs_df['delta_pec50'],
        c=cliff_pairs_df['sali'].clip(0, 20),
        cmap='RdYlGn_r',
        alpha=0.7, s=20
    )
    plt.colorbar(scatter, ax=ax, label='SALI')
    ax.axhline(DELTA_PVAL_THRESHOLD, color='red', linestyle='--', alpha=0.5)
    ax.axvline(TANIMOTO_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
    ax.set_xlabel('Tanimoto Similarity')
    ax.set_ylabel('|ΔpEC50|')
    ax.set_title('Activity Cliff Map\n(PXR Training Set)')
else:
    ax.text(0.5, 0.5, 'No cliff pairs', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Activity Cliff Map')

# (c) Test set cliff proximity distribution
ax = axes[2]
if len(prox_df) > 0 and 'cliff_proximity_score' in prox_df.columns:
    ax.hist(prox_df['cliff_proximity_score'], bins=30, color='darkorange', edgecolor='white', linewidth=0.5)
    ax.axvline(0.5, color='blue', linestyle='--', alpha=0.7, label='Tan=0.5')
    ax.axvline(0.7, color='red', linestyle='--', alpha=0.7, label='Tan=0.7')
    ax.set_xlabel('Max Tanimoto to Any Cliff Member')
    ax.set_ylabel('Test Compound Count')
    ax.set_title(f'Test Set Cliff Proximity\n(n={len(prox_df):,} test compounds)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No proximity data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Test Set Cliff Proximity')

plt.tight_layout()
fig_path = FIGURES / '38_activity_cliff_mining.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Figure saved to {fig_path}')

# Final summary
print('\n== Activity Cliff Mining Summary ==')
print(f'  Training compounds:           {len(train):,}')
print(f'  Cliff pairs found:            {len(cliff_pairs_df):,}')
print(f'  Cliff member compounds:       {n_cliff:,} / {len(train):,}')
print(f'  Cross-dataset cliff pairs:    {len(cross_cliff_df):,}')
if len(prox_df) > 0:
    n_high_prox = (prox_df['cliff_proximity_score'] > 0.5).sum()
    print(f'  Test compounds near cliffs:   {n_high_prox:,} / {len(prox_df):,} (Tan>0.5 to cliff member)')

print('\nOutput files:')
for f in [
    DATA_PROCESSED / 'cliff_labels.parquet',
    DATA_PROCESSED / 'cliff_pairs.parquet',
    DATA_PROCESSED / 'cross_cliff_pairs.parquet',
    DATA_PROCESSED / 'test_cliff_proximity.parquet',
]:
    size = f'{f.stat().st_size / 1024:.1f} KB' if f.exists() else 'missing'
    print(f'  {f.name:<40s}  {size}')

Figure saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\figures\38_activity_cliff_mining.png

== Activity Cliff Mining Summary ==
  Training compounds:           4,139
  Cliff pairs found:            149
  Cliff member compounds:       248 / 4,139
  Cross-dataset cliff pairs:    524
  Test compounds near cliffs:   32 / 513 (Tan>0.5 to cliff member)

Output files:
  cliff_labels.parquet                      233.5 KB
  cliff_pairs.parquet                       23.9 KB
  cross_cliff_pairs.parquet                 28.7 KB
  test_cliff_proximity.parquet              16.9 KB
